# Heritage AI — RAG Pipeline (Part C)
**Documents -> Vector Store -> RAG Chain with Citations -> CV Integration -> Multi-turn Memory**

This notebook is the full, self-contained implementation and demonstration of the
LangChain/RAG conversational system required for Part C, covering every requirement
in one place with real, saved outputs:

1. Loading 5+ heritage documents into a vector database
2. Retrieval-based answering with citations
3. Visual input integration (CV prediction -> LLM explanation)
4. Multi-turn conversation (3+ exchanges) with 6 full example conversations

**Run order:** run cells top to bottom. Sections 1-2 only need to be run once (they build
and save the vector store to disk); later sections can be re-run repeatedly to test
different questions or images.


## 0. Setup & Imports

In [34]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"  # use cached embedding model, avoid network re-checks

import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()  # reads OPENAI_API_KEY from .env in this same folder

# ---- CONFIG ----
DOCS_DIR = os.path.join("..", "Documents")
MODEL_PATH = os.path.join("..", "Models", "efficientnet_heritage.pth")
CLASS_NAMES = ["Baroque", "Gothic", "Neoclassical", "Roman", "Victorian"]  # confirmed order from training notebook
VECTORSTORE_DIR = "faiss_index"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
LLM_MODEL = "gpt-4o-mini"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K = 6

print("Setup complete.")

Setup complete.


## 1. Load Documents
Loads all PDFs in `Documents/` (6 heritage architecture documents) using PyPDFLoader.

In [19]:
def load_all_documents(docs_dir=DOCS_DIR):
    if not os.path.isdir(docs_dir):
        raise FileNotFoundError(f"Could not find folder: {docs_dir}")

    pdf_files = sorted(f for f in os.listdir(docs_dir) if f.lower().endswith(".pdf"))
    if len(pdf_files) == 0:
        raise FileNotFoundError(f"No PDF files found in {docs_dir}")

    print(f"Found {len(pdf_files)} PDF(s) in {docs_dir}:")
    for f in pdf_files:
        print(f"  - {f}")

    all_documents = []
    for filename in pdf_files:
        filepath = os.path.join(docs_dir, filename)
        loader = PyPDFLoader(filepath)
        pages = loader.load()
        for page in pages:
            page.metadata["source"] = filename
        all_documents.extend(pages)
        print(f"  Loaded '{filename}': {len(pages)} page(s)")

    return all_documents


documents = load_all_documents()

print("\n--- Sanity check ---")
print(f"Total pages loaded: {len(documents)}")
print(f"Total characters loaded: {sum(len(d.page_content) for d in documents):,}")
print("\n--- First page preview (first 400 characters) ---")
print(documents[0].page_content[:400])

Found 6 PDF(s) in ..\Documents:
  - Baroque_architecture.pdf
  - Conservation_and_restoration_of_cultural_property.pdf
  - Gothic_architecture.pdf
  - Neoclassical_architecture.pdf
  - Romanesque_architecture.pdf
  - victorian_Queen_Anne_architecture.pdf
  Loaded 'Baroque_architecture.pdf': 34 page(s)
  Loaded 'Conservation_and_restoration_of_cultural_property.pdf': 23 page(s)
  Loaded 'Gothic_architecture.pdf': 72 page(s)
  Loaded 'Neoclassical_architecture.pdf': 38 page(s)
  Loaded 'Romanesque_architecture.pdf': 57 page(s)
  Loaded 'victorian_Queen_Anne_architecture.pdf': 5 page(s)

--- Sanity check ---
Total pages loaded: 229
Total characters loaded: 429,293

--- First page preview (first 400 characters) ---
Baroque architecture
Clockwise from top left: Church of Saint Ignatius
of Loyola in Italy, Church of Santa Prisca de
Taxco in Mexico, Smolny Cathedral in Russia,
St-Gervais-et-St-Protais in France
Years active Late 16th–18th centuries
Location Europe and Latin America
Baroque ar

## 2. Build Vector Store
Splits documents into overlapping chunks, embeds them locally with `all-MiniLM-L6-v2`
(free, runs on CPU, no API cost), and builds a FAISS index saved to disk.

In [20]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks from {len(documents)} pages.")
print(f"Average chunk length: {sum(len(c.page_content) for c in chunks) // len(chunks)} characters")

print("\nEmbedding chunks and building FAISS index (downloads model once, then cached)...")
embeddings = HuggingFaceEmbeddings(model_name=f"sentence-transformers/{EMBEDDING_MODEL}")
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local(VECTORSTORE_DIR)
print(f"Saved vector store to ./{VECTORSTORE_DIR}/")

Created 595 chunks from 229 pages.
Average chunk length: 812 characters

Embedding chunks and building FAISS index (downloads model once, then cached)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5894.73it/s]


Saved vector store to ./faiss_index/


In [21]:
# Quick retrieval sanity check across two different styles
for test_query in [
    "What are the key features of Gothic architecture?",
    "What are the defining features of Baroque architecture?",
]:
    results = vectorstore.similarity_search(test_query, k=3)
    print(f"\nQuery: '{test_query}'")
    for i, r in enumerate(results, 1):
        print(f"[{i}] Source: {r.metadata.get('source')} (page {r.metadata.get('page')})")
        print(f"    {r.page_content[:150]}...")


Query: 'What are the key features of Gothic architecture?'
[1] Source: Gothic_architecture.pdf (page 0)
    churches, as well as abbeys, and parish churches. It is
also the architecture of many castles, palaces, town
halls, guildhalls, universities, and, les...
[2] Source: Romanesque_architecture.pdf (page 0)
    architectural style since Imperial Roman architecture.
As is the case with Gothic, the name of the style was
transferred onto the contemporary Romanes...
[3] Source: Gothic_architecture.pdf (page 60)
    Architecture portal
Architectural history
Architecture of cathedrals and great churches
Carpenter Gothic
Collegiate Gothic in North America
Gothicmed
...

Query: 'What are the defining features of Baroque architecture?'
[1] Source: Baroque_architecture.pdf (page 0)
    Baroque architecture
Clockwise from top left: Church of Saint Ignatius
of Loyola in Italy, Church of Santa Prisca de
Taxco in Mexico, Smolny Cathedral...
[2] Source: Baroque_architecture.pdf (page 29)
    throu

## 3. RAG Chain — Retrieval + LLM with Citations
Retrieves relevant chunks for a question, sends them to GPT-4o-mini, and returns
a grounded answer with citations (source filename + page).

In [22]:
RAG_PROMPT_TEMPLATE = """You are a heritage architecture assistant for the National Heritage Preservation Trust.
Answer the user's question using ONLY the context provided below. If the context
doesn't contain enough information to answer, say so honestly instead of guessing.

After your answer, list the sources you used in this exact format:
Sources: [filename1, filename2, ...]

Context:
{context}

Question: {question}

Answer:"""

llm = ChatOpenAI(model=LLM_MODEL, temperature=0.2)


def format_context_with_sources(retrieved_docs):
    context_parts, citations = [], []
    for doc in retrieved_docs:
        source = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page", "?")
        context_parts.append(f"[{source}, page {page}]\n{doc.page_content}")
        citations.append(source)
    seen = set()
    unique_citations = [c for c in citations if not (c in seen or seen.add(c))]
    return "\n\n---\n\n".join(context_parts), unique_citations


def ask_question(question, vectorstore=vectorstore, llm=llm):
    retrieved_docs = vectorstore.similarity_search(question, k=TOP_K)
    context, citations = format_context_with_sources(retrieved_docs)
    prompt = ChatPromptTemplate.from_template(RAG_PROMPT_TEMPLATE)
    messages = prompt.format_messages(context=context, question=question)
    response = llm.invoke(messages)
    return response.content, citations


print("RAG chain ready.")

RAG chain ready.


### Example questions (edit and re-run this cell for more)

In [33]:
questions = [
    "What is Gothic architecture?",
    "Give examples of Baroque architecture.",
    "Give difference between Neoclassical and roman architecture.",
]

for q in questions:
    answer, citations = ask_question(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print(f"(Retrieved from: {', '.join(citations)})")
    print("-" * 80)

Q: What is Gothic architecture?
A: Gothic architecture is a style that began in the early 12th century in northwest France and England, characterized by features such as pointed arches, ribbed vaults, and flying buttresses. It became a leading form of artistic expression during the late Middle Ages and spread throughout Latin Europe in the 13th century. By 1300, a first "International Style" of Gothic had developed. This architectural style is notable for its application in churches, abbeys, castles, palaces, town halls, and universities, with many examples listed as UNESCO World Heritage Sites.

Sources: [Gothic_architecture.pdf, page 0], [Gothic_architecture.pdf, page 3]
(Retrieved from: Gothic_architecture.pdf)
--------------------------------------------------------------------------------
Q: Give examples of Baroque architecture.
A: Examples of Baroque architecture include the Church of Saint Ignatius of Loyola in Italy, the Church of Santa Prisca de Taxco in Mexico, Smolny Cathed

## 4. CV -> RAG Integration
Loads the trained EfficientNetB0 model, runs a real prediction on an image
({style}, {confidence}), and feeds that structured output into the RAG chain
to produce a grounded, cited explanation. This is the "CV-LLM data handoff"
mechanism required by the brief.

In [24]:
def build_cv_model(num_classes=5):
    model = models.efficientnet_b0(weights=None)
    num_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(num_features, num_classes)
    return model


def load_cv_model():
    model = build_cv_model(num_classes=len(CLASS_NAMES))
    state_dict = torch.load(MODEL_PATH, map_location=torch.device("cpu"))
    model.load_state_dict(state_dict)
    model.eval()
    return model


def get_transform():
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


def predict_style(model, image_path):
    transform = get_transform()
    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0)
    with torch.no_grad():
        outputs = model(image_tensor)
        probabilities = torch.nn.functional.softmax(outputs[0], dim=0)
        confidence, predicted_idx = torch.max(probabilities, dim=0)
    return {"style": CLASS_NAMES[predicted_idx.item()], "confidence": confidence.item()}


cv_model = load_cv_model()
print("CV model loaded.")

CV model loaded.


In [25]:
INTEGRATED_PROMPT_TEMPLATE = """You are a heritage architecture assistant for the National Heritage Preservation Trust.
A computer vision model has just analyzed an uploaded image and produced this structured prediction:

    Predicted style: {predicted_style}
    Confidence: {confidence:.1%}

Using ONLY the reference context below, explain this architectural style to the user,
tailoring your explanation to what was detected. If confidence is below 60%, mention
that the prediction is uncertain. If context is insufficient, say so honestly.

After your answer, list sources in this format:
Sources: [filename1, filename2, ...]

Reference context:
{context}

Answer:"""


def explain_prediction(prediction, vectorstore=vectorstore, llm=llm):
    query = f"Key features and history of {prediction['style']} architecture"
    retrieved_docs = vectorstore.similarity_search(query, k=TOP_K)
    context, citations = format_context_with_sources(retrieved_docs)
    prompt = ChatPromptTemplate.from_template(INTEGRATED_PROMPT_TEMPLATE)
    messages = prompt.format_messages(
        predicted_style=prediction["style"], confidence=prediction["confidence"], context=context
    )
    response = llm.invoke(messages)
    return response.content, citations


# EDIT this path to point at any real image from your dataset
TEST_IMAGE_PATH = r"C:\Users\Amani\Projects\heritage-ai\dataset\Gothic\02_0002.jpg"

prediction = predict_style(cv_model, TEST_IMAGE_PATH)
print(f"Predicted style: {prediction['style']}")
print(f"Confidence: {prediction['confidence']:.2%}\n")

explanation, citations = explain_prediction(prediction)
print("--- Grounded, cited explanation ---")
print(explanation)
print(f"\n(Retrieved from: {', '.join(citations)})")

Predicted style: Gothic
Confidence: 98.68%

--- Grounded, cited explanation ---
Gothic architecture is a prominent architectural style that flourished in Europe from the late 12th century to the 16th century, particularly during the High and Late Middle Ages. It originated in the Île-de-France and Picardy regions of northern France and is characterized by its innovative structural techniques and aesthetic features.

Key elements of Gothic architecture include:

1. **Pointed Arches**: These arches not only provide structural support but also allow for taller and more slender designs compared to the rounded arches of Romanesque architecture.

2. **Ribbed Vaults**: This technique involves intersecting stone ribs that support the ceiling, enabling the creation of expansive interior spaces.

3. **Flying Buttresses**: These external supports transfer the weight of the roof and walls outward, allowing for higher ceilings and larger windows.

4. **Large Stained Glass Windows**: Gothic building

## 5. Multi-turn Conversation with Memory
The brief requires multi-turn conversation (3+ exchanges) where the assistant
remembers earlier turns. This is implemented with a simple, self-contained
memory class (no external dependency needed) that stores the conversation
history and feeds it back into each new prompt.

This demonstrates the assistant correctly resolving references like "it" or
"this style" back to something mentioned earlier in the conversation —
not just answering each question in isolation.

In [26]:
class SimpleConversationMemory:
    """Stores conversation turns and formats them for inclusion in each new prompt."""
    def __init__(self):
        self.turns = []

    def save_context(self, question, answer):
        self.turns.append((question, answer))

    @property
    def buffer_as_str(self):
        if not self.turns:
            return "(no previous conversation)"
        lines = []
        for q, a in self.turns:
            lines.append(f"User: {q}")
            lines.append(f"Assistant: {a}")
        return "\n".join(lines)


CHAT_PROMPT_TEMPLATE = """You are a heritage architecture assistant for the National Heritage Preservation Trust.
Using ONLY the reference context below, answer the user's question. If context
is insufficient, say so honestly rather than guessing.

Conversation so far:
{chat_history}

Reference context:
{context}

User question: {question}

After your answer, list sources used in this format:
Sources: [filename1, filename2, ...]

Answer:"""


def ask_with_memory(question, memory, vectorstore=vectorstore, llm=llm):
    retrieved_docs = vectorstore.similarity_search(question, k=TOP_K)
    context, citations = format_context_with_sources(retrieved_docs)
    prompt = ChatPromptTemplate.from_template(CHAT_PROMPT_TEMPLATE)
    messages = prompt.format_messages(
        chat_history=memory.buffer_as_str, context=context, question=question
    )
    response = llm.invoke(messages)
    memory.save_context(question, response.content)
    return response.content, citations


print("Multi-turn chat function ready.")

Multi-turn chat function ready.


### Example Conversation 1 — Gothic architecture (4 exchanges)
Notice the 3rd and 4th questions use "it" / "this style" without re-stating
"Gothic" — the assistant correctly resolves this from conversation memory,
proving multi-turn context retention (not just independent Q&A).

In [27]:
memory_1 = SimpleConversationMemory()

conversation_1 = [
    "What is Gothic architecture?",
    "What are its most unique characteristics?",
    "Give me 3 real examples of buildings in this style.",
    "How does it compare to Baroque architecture?",
]

for turn_num, question in enumerate(conversation_1, 1):
    answer, citations = ask_with_memory(question, memory_1)
    print(f"Turn {turn_num}")
    print(f"User: {question}")
    print(f"Assistant: {answer}")
    print(f"(Retrieved from: {', '.join(citations)})")
    print("=" * 80)

Turn 1
User: What is Gothic architecture?
Assistant: Gothic architecture is a style of architecture that originated in the early 12th century in northwest France and England, spreading throughout Latin Europe by the 13th century. It is characterized by features such as pointed arches, ribbed vaults, and flying buttresses, which allowed for taller structures and larger windows, often filled with stained glass. This architectural style became a prominent form of artistic expression during the late Middle Ages and is notably seen in churches, cathedrals, abbeys, and other significant buildings. Gothic architecture continued to evolve and influence design until the Renaissance, and it experienced several revivals from the mid-18th century onwards.

Sources: [Gothic_architecture.pdf, page 0], [Gothic_architecture.pdf, page 3]
(Retrieved from: Gothic_architecture.pdf)
Turn 2
User: What are its most unique characteristics?
Assistant: The reference context provided does not contain specific in

### Example Conversation 2 — Baroque architecture (3 exchanges)

In [28]:
memory_2 = SimpleConversationMemory()

conversation_2 = [
    "What is Baroque architecture?",
    "When and where did it originate?",
    "Name one famous example of it.",
]

for turn_num, question in enumerate(conversation_2, 1):
    answer, citations = ask_with_memory(question, memory_2)
    print(f"Turn {turn_num}")
    print(f"User: {question}")
    print(f"Assistant: {answer}")
    print(f"(Retrieved from: {', '.join(citations)})")
    print("=" * 80)

Turn 1
User: What is Baroque architecture?
Assistant: Baroque architecture is a highly decorative and theatrical style that originated in Italy in the late 16th century and gradually spread across Europe and Latin America. It was initially introduced by the Catholic Church, particularly the Jesuits, as a response to the Reformation, aiming to inspire astonishment, reverence, and awe. The style reached its peak during the High Baroque period (1625–1675) and was prominently used in churches and palaces across various countries, including Italy, Spain, Portugal, France, Bavaria, and Austria. In the Late Baroque period (1675–1750), the style extended to regions such as Russia and the Ottoman Empire, as well as the Spanish and Portuguese colonies in Latin America.

Sources: [Baroque_architecture.pdf, page 0]
(Retrieved from: Baroque_architecture.pdf)
Turn 2
User: When and where did it originate?
Assistant: Baroque architecture originated in Italy in the late 16th century. 

Sources: [Baroqu

### Example Conversation 3 — Neoclassical architecture (3 exchanges)

In [29]:
memory_3 = SimpleConversationMemory()

conversation_3 = [
    "What defines Neoclassical architecture?",
    "What historical movement influenced it?",
    "Give an example building and explain why it fits this style.",
]

for turn_num, question in enumerate(conversation_3, 1):
    answer, citations = ask_with_memory(question, memory_3)
    print(f"Turn {turn_num}")
    print(f"User: {question}")
    print(f"Assistant: {answer}")
    print(f"(Retrieved from: {', '.join(citations)})")
    print("=" * 80)

Turn 1
User: What defines Neoclassical architecture?
Assistant: Neoclassical architecture is defined by several key characteristics: it emphasizes symmetry, simple geometry, and the wall rather than ornamentation. The style arose as a reaction against the Rococo style's naturalistic ornamentation and is rooted in the classical architectural vocabulary of ancient Greece and Rome. Neoclassical architecture maintains distinct identities for each of its parts and is often associated with the Enlightenment and the study of classical antiquity. It became prominent in the Western world from the mid-18th century to the mid-19th century and has influenced various architectural styles, including the Adam style, Empire, Federal, and Greek Revival.

Sources: [Neoclassical_architecture.pdf, page 0], [Neoclassical_architecture.pdf, page 1]
(Retrieved from: Neoclassical_architecture.pdf)
Turn 2
User: What historical movement influenced it?
Assistant: Neoclassical architecture was influenced by the En

### Example Conversation 4 — Romanesque architecture (3 exchanges)

In [30]:
memory_4 = SimpleConversationMemory()

conversation_4 = [
    "What is Romanesque architecture?",
    "How does it differ from Gothic architecture?",
    "What structural features define it?",
]

for turn_num, question in enumerate(conversation_4, 1):
    answer, citations = ask_with_memory(question, memory_4)
    print(f"Turn {turn_num}")
    print(f"User: {question}")
    print(f"Assistant: {answer}")
    print(f"(Retrieved from: {', '.join(citations)})")
    print("=" * 80)

Turn 1
User: What is Romanesque architecture?
Assistant: Romanesque architecture is a style that emerged in Europe, characterized by its massive solidity and strength. It is often described as a debased form of Roman architecture, with the term "Romanesque" indicating its origins. The architecture is generally divided into two periods: the "First Romanesque," which features rubble walls, smaller windows, and unvaulted roofs, and the more refined "Romanesque" style, which includes increased use of vaults and dressed stone. Romanesque buildings are noted for their reliance on thick walls and piers rather than columns and arches, creating a distinctive appearance. This style is seen in various types of structures, including churches, castles, and civic buildings, and is recognized for its regional variations across Europe.

Sources: [Romanesque_architecture.pdf]
(Retrieved from: Romanesque_architecture.pdf)
Turn 2
User: How does it differ from Gothic architecture?
Assistant: Romanesque ar

### Example Conversation 5 — Victorian / Queen Anne style (3 exchanges)

In [31]:
memory_5 = SimpleConversationMemory()

conversation_5 = [
    "What is the Queen Anne style of Victorian architecture?",
    "What decorative features are typical of it?",
    "Where was this style most commonly built?",
]

for turn_num, question in enumerate(conversation_5, 1):
    answer, citations = ask_with_memory(question, memory_5)
    print(f"Turn {turn_num}")
    print(f"User: {question}")
    print(f"Assistant: {answer}")
    print(f"(Retrieved from: {', '.join(citations)})")
    print("=" * 80)

Turn 1
User: What is the Queen Anne style of Victorian architecture?
Assistant: The Queen Anne style of Victorian architecture is a popular architectural style that emerged in the United States from roughly 1880 to 1910. It is characterized by a wide range of picturesque buildings with "free Renaissance" details, rather than adhering to a specific formulaic style. This style replaced the French-derived Second Empire style and is noted for its asymmetrical façades, dominant front-facing gables, overhanging eaves, and various types of towers. Distinctive features may include wrap-around porches, shaped gables, and decorative elements that contribute to its picturesque quality. While the high Queen Anne style waned in popularity in the early 1900s, some elements persisted into the 1920s.

Sources: [victorian_Queen_Anne_architecture.pdf]
(Retrieved from: victorian_Queen_Anne_architecture.pdf)
Turn 2
User: What decorative features are typical of it?
Assistant: The reference context provided

### Example Conversation 6 — Heritage conservation (general context, 3 exchanges)

In [32]:
memory_6 = SimpleConversationMemory()

conversation_6 = [
    "What does conservation and restoration of cultural property involve?",
    "Why is this important for heritage sites specifically?",
    "What are common challenges in restoring historic buildings?",
]

for turn_num, question in enumerate(conversation_6, 1):
    answer, citations = ask_with_memory(question, memory_6)
    print(f"Turn {turn_num}")
    print(f"User: {question}")
    print(f"Assistant: {answer}")
    print(f"(Retrieved from: {', '.join(citations)})")
    print("=" * 80)

Turn 1
User: What does conservation and restoration of cultural property involve?
Assistant: Conservation and restoration of cultural property involve the protection and care of cultural property, which includes artworks, architecture, archaeology, and museum collections. Key activities in this field include preventive conservation, examination, documentation, research, treatment, and education. The overarching goal is to keep cultural property in as close to its original condition as possible for as long as possible, using effective methods. Ethical guidelines in conservation emphasize minimal intervention, the use of appropriate materials and reversible methods, and full documentation of all work undertaken.

Sources: [Conservation_and_restoration_of_cultural_property.pdf, page 0, page 5]
(Retrieved from: Conservation_and_restoration_of_cultural_property.pdf)
Turn 2
User: Why is this important for heritage sites specifically?
Assistant: The conservation and restoration of cultural pr

## Notes for report writing
- Section 1 confirms all 6 documents load with real, non-truncated text (429,293+ characters total)
- Section 2 confirms retrieval correctly distinguishes between styles (Gothic vs. Baroque queries return different sources)
- Section 3 satisfies the brief's "retrieval-based answering with citations" requirement
- Section 4 satisfies the "CV -> LLM data handoff" requirement, using real CV output ({style}, {confidence}) as structured input to the LLM prompt
- Section 5 satisfies the "multi-turn conversation (3+ exchanges)" requirement, with 6 full example conversations (one per architecture style, plus general heritage conservation), each showing the assistant correctly using earlier turns as context
- Together, Sections 1-5 cover every functional requirement listed under Part C of the brief: 5+ documents loaded, retrieval-based answering with citations, multi-turn conversation, and CV-LLM visual integration
